# Mastering PySpark: From Mathematician to Data Engineer

**A Definitive Guide to Apache Spark Internals, Optimization & Streaming**

---

### About this Tutorial

If you come from a **SQL/BigQuery/Pandas** background and want to truly understand *why* Spark works the way it does — not just *how* — this notebook is for you.

We'll build intuition using **mathematical analogies** (DAGs as directed graphs, columnar storage as matrix transpositions, HyperLogLog as probabilistic hashing) while running **real PySpark code** on 1 million simulated rows.

| Section | Core Concept | BigQuery / SQL Analogy |
|---|---|---|
| 1 | SparkSession & Mock Data | `CREATE TABLE` |
| 2 | Lazy Evaluation & DAGs | Query Planning in BigQuery |
| 3 | Jobs, Stages & Tasks | Slot allocation in BigQuery |
| 4 | Shuffle & State Management | `GROUP BY` redistribution |
| 5 | Catalyst Optimizer & Parquet | BigQuery optimizer + columnar format |
| 6 | UDFs vs Pandas UDFs | JS UDFs in BigQuery (avoid if possible!) |
| 7 | Streaming & Window Functions | Streaming Buffer in BigQuery |

---


## 1. Introduction & Setup

### What is Apache Spark?

Apache Spark is a **distributed computation engine**. Think of it as a system that:

1. Takes your data and **partitions** it across multiple machines (like sharding a BigQuery table).
2. Builds an **execution plan** (like BigQuery's query plan) as a DAG.
3. Executes the plan **in parallel** across all machines.

The key difference from BigQuery: **you control the plan**. You decide partitioning, caching, and execution strategy. With great power comes great responsibility.

### Installing PySpark (Kaggle already has it!)


In [1]:
# If running locally, uncomment:
# !pip install pyspark==3.5.8 pandas pyarrow

# ⚠️ IMPORTANT: Environment variables MUST be set BEFORE importing pyspark.
# The JVM reads HADOOP_HOME at startup; if it's not set, file I/O fails on Windows.
import os, sys, tempfile, warnings, zipfile
warnings.filterwarnings('ignore')

if sys.platform == "win32":
    os.environ["JAVA_HOME"]      = r"C:\Program Files\Microsoft\jdk-17.0.18.8-hotspot"
    os.environ["HADOOP_HOME"]    = r"C:\hadoop"
    os.environ["PYSPARK_PYTHON"] = sys.executable   # Tell Spark which Python to use for workers
    os.environ["PATH"] = (
        os.environ["JAVA_HOME"] + r"\bin;" +
        os.environ["HADOOP_HOME"] + r"\bin;" +
        os.environ["PATH"]
    )
    # Verify winutils exists
    winutils = os.path.join(os.environ["HADOOP_HOME"], "bin", "winutils.exe")
    assert os.path.isfile(winutils), f"winutils.exe not found at {winutils}"
    print(f"JAVA_HOME       = {os.environ['JAVA_HOME']}")
    print(f"HADOOP_HOME     = {os.environ['HADOOP_HOME']}")
    print(f"PYSPARK_PYTHON  = {os.environ['PYSPARK_PYTHON']}")
    print(f"winutils        = {winutils}")

    # ── Windows-only fix: PySpark's worker.py does not flush() the socket ──
    # On Windows the buffered socket may not flush before the process exits,
    # causing "Python worker exited unexpectedly (crashed)" / EOFException.
    # We patch the bundled pyspark.zip to add explicit outfile.flush() calls.
    import pyspark as _ps
    _zip = os.path.join(os.path.dirname(_ps.__file__), "python", "lib", "pyspark.zip")
    _marker = "# WINDOWS FIX: flush buffered socket"
    with zipfile.ZipFile(_zip, "r") as _zr:
        _w = _zr.read("pyspark/worker.py").decode()
    if _marker not in _w:
        print("Applying Windows socket-flush fix to pyspark worker...")
        with zipfile.ZipFile(_zip, "r") as _zr:
            _names = _zr.namelist()
            _all = {n: _zr.read(n) for n in _names}
        # Fix 1: flush after dump_stream
        _w = _w.replace(
            "serializer.dump_stream(out_iter, outfile)\n            finally:",
            "serializer.dump_stream(out_iter, outfile)\n"
            "                outfile.flush()  " + _marker + "\n"
            "            finally:",
            1,
        )
        # Fix 2: flush after END_OF_STREAM
        _w = _w.replace(
            "write_int(SpecialLengths.END_OF_STREAM, outfile)\n    else:",
            "write_int(SpecialLengths.END_OF_STREAM, outfile)\n"
            "        outfile.flush()  " + _marker + "\n"
            "    else:",
            1,
        )
        _all["pyspark/worker.py"] = _w.encode()
        with zipfile.ZipFile(_zip, "w", zipfile.ZIP_DEFLATED) as _zw:
            for _n in _names:
                _zw.writestr(_n, _all[_n])
        print("  -> pyspark.zip patched OK")
    else:
        print("Worker flush fix already applied.")
    del _ps, _zip, _marker, _w

# Cross-platform temp directory for Parquet/CSV demos
TEMP_DIR = os.path.join(tempfile.gettempdir(), "spark_tutorial")
os.makedirs(TEMP_DIR, exist_ok=True)
print(f"TEMP_DIR        = {TEMP_DIR}")


JAVA_HOME       = C:\Program Files\Microsoft\jdk-17.0.18.8-hotspot
HADOOP_HOME     = C:\hadoop
PYSPARK_PYTHON  = c:\Users\carlo\AppData\Local\Programs\Python\Python312\python.exe
winutils        = C:\hadoop\bin\winutils.exe
Worker flush fix already applied.
TEMP_DIR        = C:\Users\carlo\AppData\Local\Temp\spark_tutorial


### 1.1 Starting a SparkSession

The `SparkSession` is the **single entry point** to all Spark functionality. In BigQuery terms, it's like opening the BigQuery console — it's your connection to the engine.

Under the hood, `SparkSession` wraps:
- **SparkContext** → connection to the cluster
- **SQLContext** → SQL query interface
- **HiveContext** → metastore access

Since Spark 2.0, you only need `SparkSession`. Think of it as the **universal constructor**.


In [2]:
from pyspark.sql import SparkSession
import time

_builder = (
    SparkSession.builder
    .master("local[*]")                           # Use all available CPU cores
    .appName("Mastering PySpark Tutorial")         # Name visible in Spark UI
    .config("spark.sql.shuffle.partitions", 8)     # Reduce shuffle partitions for local mode
    .config("spark.driver.memory", "4g")           # Allocate memory to the driver
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")  # Enable Apache Arrow
    .config("spark.python.worker.reuse", "true")   # Reuse Python workers (faster UDFs)
    .config("spark.python.worker.timeout", "120")  # 120s timeout for Python workers (Windows)
)
# Windows: explicitly tell Spark which Python to use & disable daemon (SIGHUP unsupported)
if sys.platform == "win32":
    _builder = (
        _builder
        .config("spark.pyspark.python", sys.executable)
        .config("spark.pyspark.driver.python", sys.executable)
        .config("spark.python.use.daemon", "false")
    )
spark = _builder.getOrCreate()

# CRITICAL on Windows: set hadoop.home.dir as a JVM system property DIRECTLY.
# In local mode, spark.driver.extraJavaOptions has NO effect (the JVM is already running).
# We must set it via the live JVM before any Hadoop class (Shell) is loaded.
if sys.platform == "win32":
    spark._jvm.java.lang.System.setProperty(
        "hadoop.home.dir", os.environ["HADOOP_HOME"].replace("\\", "/")
    )
    print(f"✅ JVM hadoop.home.dir = {spark._jvm.java.lang.System.getProperty('hadoop.home.dir')}")

# Suppress noisy Spark logs
spark.sparkContext.setLogLevel("ERROR")

# Verify the session
print(f"Spark Version : {spark.version}")
print(f"App Name      : {spark.sparkContext.appName}")
print(f"Master        : {spark.sparkContext.master}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")


✅ JVM hadoop.home.dir = C:/hadoop
Spark Version : 3.5.8
App Name      : Mastering PySpark Tutorial
Master        : local[*]
Default Parallelism: 12


> **💡 Pro Tip (Interview):** Always mention that `local[*]` means "use all cores on the local machine". In production, you'd use `yarn`, `k8s`, or `mesos` as the master. The number of cores directly maps to the **maximum parallelism** of your application.


### 1.2 Generating Mock Data: "Vitaly Health Services"

We'll simulate a healthcare dataset with **1 million rows** — enough to feel the difference between optimized and unoptimized code.

| Column | Type | Description |
|---|---|---|
| `id_paciente` | Integer | Unique patient ID |
| `provincia` | String | Province (50 Spanish provinces) |
| `coste_servicio` | Double | Cost of medical service (10–500€) |
| `fecha_cita` | Timestamp | Appointment date (last 365 days) |


In [3]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType, DoubleType, TimestampType
)

NUM_ROWS = 1_000_000

# Spanish provinces for realistic data
PROVINCIAS = [
    "Madrid", "Barcelona", "Valencia", "Sevilla", "Zaragoza", "Malaga",
    "Murcia", "Palma", "Las Palmas", "Bilbao", "Alicante", "Cordoba",
    "Valladolid", "Vigo", "Gijon", "Granada", "Elche", "Oviedo",
    "Santander", "Pamplona", "Toledo", "Burgos", "Salamanca", "Albacete",
    "Cadiz", "Huelva", "Leon", "Tarragona", "Lleida", "Jaen",
    "Ourense", "Girona", "Lugo", "Caceres", "Badajoz", "Huesca",
    "Teruel", "Soria", "Segovia", "Avila", "Cuenca", "Guadalajara",
    "Ciudad Real", "Zamora", "Palencia", "Pontevedra", "La Coruna",
    "Almeria", "Castellon", "Logrono"
]

# ---------- Generate 1M rows using Spark (not Pandas!) ----------
# This is the "Spark way": generate data in parallel using the distributed engine.

start_time = time.time()

df = (
    spark.range(0, NUM_ROWS)                                       # Generates IDs 0 to 999,999
    .withColumn("id_paciente", (F.col("id") + 1).cast(IntegerType()))
    .withColumn(
        "provincia",
        F.array(*[F.lit(p) for p in PROVINCIAS])                   # Create array of provinces
         .getItem((F.rand() * len(PROVINCIAS)).cast(IntegerType()))  # Random pick
    )
    .withColumn(
        "coste_servicio",
        F.round(F.rand() * 490 + 10, 2)                           # Uniform [10, 500]
    )
    .withColumn(
        "fecha_cita",
        F.date_sub(F.current_date(), (F.rand() * 365).cast(IntegerType())).cast(TimestampType())
    )
    .drop("id")
)

# Force materialization to measure generation time
df.cache()          # Keep in memory for the rest of the tutorial
row_count = df.count()

elapsed = time.time() - start_time
print(f"✅ Generated {row_count:,} rows in {elapsed:.2f}s")
print(f"✅ Partitions: {df.rdd.getNumPartitions()}")
df.printSchema()
df.show(5, truncate=False)


✅ Generated 1,000,000 rows in 5.43s
✅ Partitions: 12
root
 |-- id_paciente: integer (nullable = false)
 |-- provincia: string (nullable = true)
 |-- coste_servicio: double (nullable = true)
 |-- fecha_cita: timestamp (nullable = true)

+-----------+----------+--------------+-------------------+
|id_paciente|provincia |coste_servicio|fecha_cita         |
+-----------+----------+--------------+-------------------+
|1          |Oviedo    |88.91         |2025-06-30 00:00:00|
|2          |Valladolid|89.65         |2025-05-15 00:00:00|
|3          |Vigo      |327.55        |2026-01-06 00:00:00|
|4          |Logrono   |42.18         |2025-04-03 00:00:00|
|5          |Palencia  |142.23        |2025-05-22 00:00:00|
+-----------+----------+--------------+-------------------+
only showing top 5 rows



---

## 2. Core Concepts: Lazy Evaluation & DAGs

### 2.1 What is Lazy Evaluation?

In mathematics, when you define a function:

$$f(x) = 3x^2 + 2x + 1$$

...nothing is computed. The function **exists as a definition**, an abstract recipe. It's only when you **evaluate** it at a specific point — $f(5)$ — that you get a number.

**Spark works exactly the same way:**

| Math | Spark |
|---|---|
| Define $f(x) = 3x^2 + 2x + 1$ | `df.filter(...).select(...)` → just builds the recipe |
| Evaluate $f(5) = 86$ | `.show()`, `.count()`, `.collect()` → triggers execution |
| The definition (recipe) | **Transformation** |
| The evaluation (result) | **Action** |

In BigQuery, this is analogous to how the query plan is built *before* execution. But in BigQuery, you don't see this separation — you hit "Run" and it does everything. In Spark, the separation is **explicit and intentional**.

### Why is this powerful?

Because Spark can **see the entire recipe** before cooking. This allows the optimizer (Catalyst) to:
- Reorder operations for efficiency
- Eliminate unnecessary computations
- Fuse multiple steps into one

### 2.2 Transformations vs. Actions

| Type | Examples | What happens |
|---|---|---|
| **Transformation** | `filter`, `select`, `groupBy`, `join`, `withColumn` | Builds the DAG (lazy) |
| **Action** | `show`, `count`, `collect`, `write`, `take` | Triggers execution |


In [4]:
# ============================================================
# DEMO: Lazy Evaluation in action
# ============================================================

# --- Step 1: Define transformations (NOTHING happens yet) ---
print("Defining transformations...")

df_filtered = (
    df
    .filter(F.col("provincia") == "Madrid")          # Narrow transformation
    .filter(F.col("coste_servicio") > 100)            # Another narrow transformation
    .select("id_paciente", "coste_servicio")          # Projection (like SELECT in SQL)
)

print("Transformations defined. Nothing has been computed yet!")
print(f"Type of df_filtered: {type(df_filtered)}")
print("\n--- No Spark job was triggered above. Let's verify: ---")

# --- Step 2: Trigger an ACTION ---
start = time.time()
count = df_filtered.count()  # THIS triggers the full execution
elapsed = time.time() - start

print(f"\n⚡ Action triggered! Found {count:,} rows in {elapsed:.4f}s")
print("Only NOW did Spark actually read and filter the data.")


Defining transformations...
Transformations defined. Nothing has been computed yet!
Type of df_filtered: <class 'pyspark.sql.dataframe.DataFrame'>

--- No Spark job was triggered above. Let's verify: ---

⚡ Action triggered! Found 16,442 rows in 0.4195s
Only NOW did Spark actually read and filter the data.


### 2.3 The DAG: Directed Acyclic Graph

When you chain transformations, Spark builds a **DAG** — a directed acyclic graph where:

- Each **node** is a transformation (filter, select, groupBy...)
- Each **edge** represents data flow from one transformation to the next
- **Acyclic** means no loops (data flows in one direction only)

Formally, the DAG is a pair $G = (V, E)$ where:
- $V = \{v_1, v_2, ..., v_n\}$ are the transformations
- $E \subseteq V \times V$ with $(v_i, v_j) \in E$ meaning $v_i$ feeds into $v_j$
- The constraint: there is no sequence $v_1 \to v_2 \to ... \to v_k \to v_1$

For our example above:

```
  [Read cached df]
        │
        ▼
  [Filter: provincia = 'Madrid']     ← Narrow
        │
        ▼
  [Filter: coste_servicio > 100]     ← Narrow
        │
        ▼
  [Select: id_paciente, coste]       ← Narrow
        │
        ▼
  [count()]                          ← ACTION → triggers everything
```

All transformations above are **narrow** (no data movement between partitions), so Spark can execute them in a **single stage**.

> **💡 Pro Tip (Interview):** "The DAG is Spark's execution blueprint. Catalyst optimizes the *logical plan* (what to compute), and Tungsten optimizes the *physical plan* (how to compute it). This two-phase optimization is what makes Spark fast."


In [5]:
# ============================================================
# Inspect the Execution Plan (like BigQuery's EXPLAIN)
# ============================================================

# The "explain" method shows you the physical plan Spark will use.
# This is the Spark equivalent of BigQuery's "Execution Details" tab.

print("=" * 60)
print("PHYSICAL PLAN (how Spark will actually execute this):")
print("=" * 60)
df_filtered.explain(mode="simple")

print("\n" + "=" * 60)
print("FORMATTED PLAN (easier to read):")
print("=" * 60)
df_filtered.explain(mode="formatted")


PHYSICAL PLAN (how Spark will actually execute this):
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [id_paciente#2, coste_servicio#9]
   +- Filter (((isnotnull(provincia#5) AND isnotnull(coste_servicio#9)) AND (provincia#5 = Madrid)) AND (coste_servicio#9 > 100.0))
      +- InMemoryTableScan [coste_servicio#9, id_paciente#2, provincia#5], [isnotnull(provincia#5), isnotnull(coste_servicio#9), (provincia#5 = Madrid), (coste_servicio#9 > 100.0)]
            +- InMemoryRelation [id_paciente#2, provincia#5, coste_servicio#9, fecha_cita#14], StorageLevel(disk, memory, deserialized, 1 replicas)
                  +- *(1) Project [id_paciente#2, provincia#5, coste_servicio#9, cast(date_sub(2026-02-14, cast((rand(5241822155580773806) * 365.0) as int)) as timestamp) AS fecha_cita#14]
                     +- *(1) Project [id_paciente#2, provincia#5, round(((rand(-4186340494610920368) * 490.0) + 10.0), 2) AS coste_servicio#9]
                        +- *(1) Project [cast((id#0L

---

## 3. Architecture: Jobs, Stages & Tasks

### 3.1 The Execution Hierarchy

When an **action** triggers execution, Spark creates:

```
Application (your SparkSession)
 └── Job 1 (triggered by .count())
      ├── Stage 0 (all narrow transformations before a shuffle)
      │    ├── Task 0 (processes Partition 0)
      │    ├── Task 1 (processes Partition 1)
      │    └── Task N (processes Partition N)
      └── Stage 1 (after shuffle boundary)
           ├── Task 0
           └── Task M
```

### The Golden Rule: **1 Partition = 1 Task**

This is the most important mental model in Spark:

- Each partition is an independent chunk of data
- Each task processes exactly one partition
- Tasks run in parallel across cores

If you have 8 partitions and 8 cores, all 8 tasks run simultaneously. If you have 1000 partitions and 8 cores, tasks run in waves of 8.

**BigQuery analogy:** In BigQuery, you don't control slots directly. Spark gives you that control via partitioning.

### 3.2 Narrow vs Wide Transformations

| Type | Data Movement | Examples | Stage Boundary? |
|---|---|---|---|
| **Narrow** | None (each partition is self-contained) | `filter`, `select`, `map`, `union` | No |
| **Wide** | **Shuffle** (data moves between partitions) | `groupBy`, `join`, `repartition`, `distinct` | **Yes** |

Mathematically:
- **Narrow**: $f: P_i \to P_i'$ — each partition maps to exactly one output partition
- **Wide**: $f: \{P_1, ..., P_n\} \to \{P_1', ..., P_m'\}$ — output partitions depend on multiple input partitions


In [6]:
# ============================================================
# DEMO: Partitions, Narrow transforms & Stage boundaries
# ============================================================

print("=== Partition Analysis ===")
print(f"Original DataFrame partitions: {df.rdd.getNumPartitions()}")

# Narrow transformations: partition count stays the same
df_narrow = df.filter(F.col("coste_servicio") > 200).select("id_paciente", "provincia")
print(f"After filter + select (narrow): {df_narrow.rdd.getNumPartitions()} partitions")

# Wide transformation: repartition changes partition count
df_repartitioned = df.repartition(16)
print(f"After repartition(16) (wide):   {df_repartitioned.rdd.getNumPartitions()} partitions")

# Coalesce: reduces partitions WITHOUT a full shuffle (narrow!)
df_coalesced = df_repartitioned.coalesce(4)
print(f"After coalesce(4) (narrow):     {df_coalesced.rdd.getNumPartitions()} partitions")

print("\n=== Key Insight ===")
print("""
- repartition(N): FULL shuffle. Use when you need to INCREASE partitions or rebalance.
- coalesce(N):    NO full shuffle. Use when you need to DECREASE partitions.
  Think of coalesce as 'merging adjacent partitions' vs repartition as 'reshuffling all data'.
""")


=== Partition Analysis ===
Original DataFrame partitions: 12
After filter + select (narrow): 12 partitions
After repartition(16) (wide):   16 partitions
After coalesce(4) (narrow):     4 partitions

=== Key Insight ===

- repartition(N): FULL shuffle. Use when you need to INCREASE partitions or rebalance.
- coalesce(N):    NO full shuffle. Use when you need to DECREASE partitions.
  Think of coalesce as 'merging adjacent partitions' vs repartition as 'reshuffling all data'.



In [7]:
# ============================================================
# DEMO: Visualize stage boundaries via explain()
# ============================================================

# This query has BOTH narrow and wide transformations
df_with_stages = (
    df
    .filter(F.col("coste_servicio") > 100)               # Narrow (Stage 0)
    .groupBy("provincia")                                  # Wide -> SHUFFLE boundary
    .agg(
        F.count("*").alias("num_pacientes"),               # Stage 1
        F.round(F.avg("coste_servicio"), 2).alias("avg_coste")
    )
    .orderBy(F.desc("num_pacientes"))                      # Wide -> another SHUFFLE
)

print("EXECUTION PLAN (notice the Exchange nodes = Shuffle boundaries):")
print("=" * 70)
df_with_stages.explain(mode="formatted")

# Trigger execution
print("\nTop 10 provinces by patient count (coste > 100):")
df_with_stages.show(10, truncate=False)


EXECUTION PLAN (notice the Exchange nodes = Shuffle boundaries):
== Physical Plan ==
AdaptiveSparkPlan (13)
+- Sort (12)
   +- Exchange (11)
      +- HashAggregate (10)
         +- Exchange (9)
            +- HashAggregate (8)
               +- Filter (7)
                  +- InMemoryTableScan (1)
                        +- InMemoryRelation (2)
                              +- * Project (6)
                                 +- * Project (5)
                                    +- * Project (4)
                                       +- * Range (3)


(1) InMemoryTableScan
Output [2]: [provincia#5, coste_servicio#9]
Arguments: [provincia#5, coste_servicio#9], [isnotnull(coste_servicio#9), (coste_servicio#9 > 100.0)]

(2) InMemoryRelation
Arguments: [id_paciente#2, provincia#5, coste_servicio#9, fecha_cita#14], CachedRDDBuilder(org.apache.spark.sql.execution.columnar.DefaultCachedBatchSerializer@1a38e60a,StorageLevel(disk, memory, deserialized, 1 replicas),*(1) Project [id_paciente#2, provin

> **💡 Pro Tip (Interview):** "Every `Exchange` node in the physical plan means a **shuffle**. Shuffles are the most expensive operation in Spark because they involve disk I/O, network transfer, and serialization. Minimizing shuffles is the #1 optimization strategy."


---

## 4. The Shuffle & State Management

### 4.1 What is a Shuffle?

A shuffle is the **physical redistribution of data across partitions** (and potentially across machines). It's the most expensive operation in distributed computing.

When you write:
```python
df.groupBy("provincia").agg(sum("coste_servicio"))
```

Spark must ensure that **all rows for "Madrid" end up on the same machine** so it can compute the sum. This means:

1. **Map side:** Each partition scans its rows and writes them to local disk, organized by the target partition (determined by `hash(provincia) % num_partitions`)
2. **Shuffle:** Data is transferred over the network to the correct machines
3. **Reduce side:** Each partition now has all rows for its assigned provinces and computes the aggregation

### The Cost Model

For $N$ rows distributed across $P$ partitions with $K$ distinct keys:

$$\text{Shuffle cost} \propto N \times (\text{serialization} + \text{network I/O} + \text{disk I/O} + \text{deserialization})$$

This is why shuffles dominate execution time for large datasets.

### 4.2 State Management: Breaking "Shared Nothing"

Spark follows a **"Shared Nothing" architecture**: each task is independent and doesn't share memory with others. But aggregations *require* combining partial results from different machines.

Spark handles this with a **two-phase aggregation**:

1. **Partial aggregation** (Map side): Each partition computes a *local* sum for its provinces
2. **Final aggregation** (Reduce side): After shuffle, local sums are combined into the final result

This is exactly the **MapReduce paradigm**: `Map → Shuffle → Reduce`.

In BigQuery, the same thing happens behind the scenes when you `GROUP BY` — slots perform partial aggregations, then results are combined. The difference: BigQuery manages this automatically; in Spark, you can tune it.


In [8]:
# ============================================================
# DEMO: Shuffle in Action — groupBy aggregation
# ============================================================

print("=== Before Shuffle ===")
print(f"Input partitions: {df.rdd.getNumPartitions()}")

start = time.time()

# This triggers a SHUFFLE: all rows for the same provincia
# must be moved to the same partition.
df_agg = (
    df
    .groupBy("provincia")
    .agg(
        F.count("*").alias("total_citas"),
        F.round(F.sum("coste_servicio"), 2).alias("coste_total"),
        F.round(F.avg("coste_servicio"), 2).alias("coste_medio"),
        F.round(F.stddev("coste_servicio"), 2).alias("coste_stddev")
    )
)

# Force execution
result = df_agg.orderBy(F.desc("coste_total"))
result.show(10, truncate=False)

elapsed = time.time() - start
print(f"\n=== After Shuffle ===")
print(f"Output partitions: {df_agg.rdd.getNumPartitions()}")
print(f"Execution time: {elapsed:.2f}s")
print(f"\nNote: shuffle.partitions = {spark.conf.get('spark.sql.shuffle.partitions')}")
print("This controls how many output partitions a shuffle creates.")


=== Before Shuffle ===
Input partitions: 12
+---------+-----------+-----------+-----------+------------+
|provincia|total_citas|coste_total|coste_medio|coste_stddev|
+---------+-----------+-----------+-----------+------------+
|Huelva   |20255      |5185965.79 |256.03     |142.05      |
|Caceres  |20212      |5167683.49 |255.67     |141.21      |
|Granada  |20058      |5147717.72 |256.64     |141.82      |
|Logrono  |20086      |5147674.59 |256.28     |141.53      |
|Castellon|20212      |5145667.17 |254.58     |141.77      |
|Badajoz  |20036      |5143094.28 |256.69     |141.12      |
|Albacete |20102      |5139334.79 |255.66     |141.96      |
|Jaen     |20197      |5136644.19 |254.33     |141.81      |
|Lleida   |20086      |5127553.58 |255.28     |142.17      |
|Lugo     |20160      |5126263.2  |254.28     |140.59      |
+---------+-----------+-----------+-----------+------------+
only showing top 10 rows


=== After Shuffle ===
Output partitions: 1
Execution time: 0.58s

Note: shu

In [9]:
# ============================================================
# DEMO: Inspect partial aggregation in the plan
# ============================================================

print("Look for 'HashAggregate' appearing TWICE in the plan:")
print("  1st HashAggregate = Partial aggregation (before shuffle)")
print("  2nd HashAggregate = Final aggregation (after shuffle)")
print("=" * 70)

df_agg.explain(mode="formatted")


Look for 'HashAggregate' appearing TWICE in the plan:
  1st HashAggregate = Partial aggregation (before shuffle)
  2nd HashAggregate = Final aggregation (after shuffle)
== Physical Plan ==
AdaptiveSparkPlan (16)
+- == Final Plan ==
   * HashAggregate (12)
   +- AQEShuffleRead (11)
      +- ShuffleQueryStage (10), Statistics(sizeInBytes=47.9 KiB, rowCount=600)
         +- Exchange (9)
            +- * HashAggregate (8)
               +- TableCacheQueryStage (7), Statistics(sizeInBytes=29.5 MiB, rowCount=1.00E+6)
                  +- InMemoryTableScan (1)
                        +- InMemoryRelation (2)
                              +- * Project (6)
                                 +- * Project (5)
                                    +- * Project (4)
                                       +- * Range (3)
+- == Initial Plan ==
   HashAggregate (15)
   +- Exchange (14)
      +- HashAggregate (13)
         +- InMemoryTableScan (1)
               +- InMemoryRelation (2)
                     +-

> **💡 Pro Tip (Interview):** "In a Spark interview, always mention that `groupBy` triggers a shuffle and that Spark uses **two-phase aggregation** (partial + final) to minimize data transfer. If asked about optimization, mention reducing shuffle partitions (`spark.sql.shuffle.partitions`) and using **salting** for skewed keys."


---

## 5. Optimization: Catalyst, Tungsten & Columnar Storage

### 5.1 The Catalyst Optimizer

Catalyst is Spark SQL's **query optimizer**. It transforms your logical plan through a series of rule-based and cost-based optimizations.

The optimization pipeline:

```
SQL / DataFrame API
       │
       ▼
  Unresolved Logical Plan    (what you wrote)
       │ ← Analysis (resolve names, types)
       ▼
  Resolved Logical Plan
       │ ← Optimization (Catalyst rules)
       ▼
  Optimized Logical Plan
       │ ← Physical Planning
       ▼
  Physical Plan(s)           (how to execute)
       │ ← Cost Model selects best
       ▼
  Selected Physical Plan
       │ ← Code Generation (Tungsten / Whole-Stage CodeGen)
       ▼
  RDDs of Internal Rows      (actual execution)
```

### Key Optimizations:

1. **Predicate Pushdown**: Filters are pushed as close to the data source as possible (like BigQuery scanning only relevant partitions)
2. **Column Pruning**: Only reads the columns you actually use (like `SELECT col1, col2` instead of `SELECT *`)
3. **Constant Folding**: Pre-computes constant expressions at planning time
4. **Filter Reordering**: Puts the most selective filters first


In [10]:
# ============================================================
# DEMO: Predicate Pushdown & Column Pruning
# ============================================================

# Let's write and read Parquet to demonstrate predicate pushdown
PARQUET_PATH = os.path.join(TEMP_DIR, "vitaly_health_data.parquet")

# Write data as Parquet (columnar format)
df.write.mode("overwrite").parquet(PARQUET_PATH)
print(f"✅ Data written to {PARQUET_PATH}")

# Now read it back with filters
df_parquet = spark.read.parquet(PARQUET_PATH)

# This query uses BOTH predicate pushdown AND column pruning
df_optimized = (
    df_parquet
    .filter(F.col("provincia") == "Barcelona")
    .filter(F.col("coste_servicio") > 300)
    .select("id_paciente", "coste_servicio")
)

print("\n=== Optimized Plan (notice PushedFilters and ReadSchema) ===")
print("PushedFilters: filters pushed INTO the Parquet reader (skips row groups!)")
print("ReadSchema: only columns actually needed are read from disk")
print("=" * 70)
df_optimized.explain(mode="formatted")


✅ Data written to C:\Users\carlo\AppData\Local\Temp\spark_tutorial\vitaly_health_data.parquet

=== Optimized Plan (notice PushedFilters and ReadSchema) ===
PushedFilters: filters pushed INTO the Parquet reader (skips row groups!)
ReadSchema: only columns actually needed are read from disk
== Physical Plan ==
* Project (4)
+- * Filter (3)
   +- * ColumnarToRow (2)
      +- Scan parquet  (1)


(1) Scan parquet 
Output [3]: [id_paciente#1469, provincia#1470, coste_servicio#1471]
Batched: true
Location: InMemoryFileIndex [file:/C:/Users/carlo/AppData/Local/Temp/spark_tutorial/vitaly_health_data.parquet]
PushedFilters: [IsNotNull(provincia), IsNotNull(coste_servicio), EqualTo(provincia,Barcelona), GreaterThan(coste_servicio,300.0)]
ReadSchema: struct<id_paciente:int,provincia:string,coste_servicio:double>

(2) ColumnarToRow [codegen id : 1]
Input [3]: [id_paciente#1469, provincia#1470, coste_servicio#1471]

(3) Filter [codegen id : 1]
Input [3]: [id_paciente#1469, provincia#1470, coste_serv

In [11]:
# ============================================================
# DEMO: Catalyst combines and reorders filters
# ============================================================

# Even if you write filters in a "bad" order, Catalyst will optimize
df_unoptimized_query = (
    df_parquet
    .select("*")                                    # SELECT * (wasteful!)
    .filter(F.col("coste_servicio") > 400)          # Filter after select
    .filter(F.col("provincia") == "Sevilla")        # Another filter
    .select("id_paciente", "provincia")             # Then project
)

print("Even with 'bad' query structure, Catalyst optimizes the plan:")
print("(Filters are pushed down, columns are pruned)")
print("=" * 70)
df_unoptimized_query.explain(mode="formatted")


Even with 'bad' query structure, Catalyst optimizes the plan:
(Filters are pushed down, columns are pruned)
== Physical Plan ==
* Project (4)
+- * Filter (3)
   +- * ColumnarToRow (2)
      +- Scan parquet  (1)


(1) Scan parquet 
Output [3]: [id_paciente#1469, provincia#1470, coste_servicio#1471]
Batched: true
Location: InMemoryFileIndex [file:/C:/Users/carlo/AppData/Local/Temp/spark_tutorial/vitaly_health_data.parquet]
PushedFilters: [IsNotNull(coste_servicio), IsNotNull(provincia), GreaterThan(coste_servicio,400.0), EqualTo(provincia,Sevilla)]
ReadSchema: struct<id_paciente:int,provincia:string,coste_servicio:double>

(2) ColumnarToRow [codegen id : 1]
Input [3]: [id_paciente#1469, provincia#1470, coste_servicio#1471]

(3) Filter [codegen id : 1]
Input [3]: [id_paciente#1469, provincia#1470, coste_servicio#1471]
Condition : (((isnotnull(coste_servicio#1471) AND isnotnull(provincia#1470)) AND (coste_servicio#1471 > 400.0)) AND (provincia#1470 = Sevilla))

(4) Project [codegen id : 1]

### 5.2 Columnar Storage: The Matrix Transposition Analogy

Traditional databases store data **row by row** (like reading a book left to right):

```
Row-oriented (CSV, MySQL, etc.):
  [id=1, provincia="Madrid", coste=150.0, fecha="2025-01-15"]
  [id=2, provincia="Barcelona", coste=230.5, fecha="2025-02-20"]
  [id=3, provincia="Madrid", coste=89.0, fecha="2025-03-10"]
```

Columnar formats (Parquet, ORC, BigQuery's internal format) store data **column by column**:

```
Column-oriented (Parquet):
  id:        [1, 2, 3, ...]
  provincia: ["Madrid", "Barcelona", "Madrid", ...]
  coste:     [150.0, 230.5, 89.0, ...]
  fecha:     ["2025-01-15", "2025-02-20", "2025-03-10", ...]
```

### The Matrix Analogy

Think of your data as a matrix $A \in \mathbb{R}^{n \times m}$ where $n$ = rows, $m$ = columns.

- **Row-oriented storage** = storing $A$ in row-major order
- **Columnar storage** = storing $A^T$ (the transpose!) in row-major order

$$A = \begin{pmatrix} a_{11} & a_{12} & \cdots & a_{1m} \\ a_{21} & a_{22} & \cdots & a_{2m} \\ \vdots & & \ddots & \vdots \\ a_{n1} & a_{n2} & \cdots & a_{nm} \end{pmatrix} \xrightarrow{\text{transpose}} A^T$$

**Why does this matter?**

1. **Column pruning:** If you only need column $j$, you read one contiguous block instead of scanning every row
2. **Compression:** Values in the same column have the same type and similar distributions → much better compression ratios (e.g., `provincia` column compresses beautifully with dictionary encoding)
3. **Vectorized operations:** CPUs can apply SIMD (Single Instruction, Multiple Data) to contiguous same-type arrays

BigQuery uses a proprietary columnar format called **Capacitor**. It's conceptually identical to Parquet.

### 5.3 The Photon Engine & Vectorized Execution

On **Databricks**, the Photon engine replaces parts of the JVM-based Spark executor with **native C++ code**. This enables:

- **Vectorized execution**: Processing data in batches of ~1024 rows at a time (using CPU vector registers)
- **Cache-friendly memory access**: Columnar data fits nicely in L1/L2 CPU cache
- **No JVM overhead**: No garbage collection pauses

This is similar to how BigQuery's Dremel engine processes data in its custom C++ execution engine.

> **💡 Pro Tip (Interview):** "When asked about Spark performance, mention the trifecta: (1) Catalyst for logical optimization, (2) Tungsten for memory management and code generation, (3) Parquet/columnar format for I/O optimization. On Databricks, add Photon for native C++ execution."


In [12]:
# ============================================================
# DEMO: Row vs Columnar Read Performance
# ============================================================

CSV_PATH = os.path.join(TEMP_DIR, "vitaly_health_data.csv")

# Write as CSV (row-oriented)
df.write.mode("overwrite").option("header", "true").csv(CSV_PATH)
print("✅ Data written as CSV and Parquet. Let's compare reads...\n")

# --- Read from CSV (row-oriented): must read ALL columns ---
start = time.time()
count_csv = (
    spark.read.option("header", "true").csv(CSV_PATH)
    .filter(F.col("provincia") == "Valencia")
    .select("id_paciente")
    .count()
)
time_csv = time.time() - start

# --- Read from Parquet (columnar): reads only needed columns ---
start = time.time()
count_parquet = (
    spark.read.parquet(PARQUET_PATH)
    .filter(F.col("provincia") == "Valencia")
    .select("id_paciente")
    .count()
)
time_parquet = time.time() - start

print(f"CSV (row-oriented):     {count_csv:,} rows in {time_csv:.3f}s")
print(f"Parquet (columnar):     {count_parquet:,} rows in {time_parquet:.3f}s")
print(f"Speedup:                {time_csv / max(time_parquet, 0.001):.1f}x faster with Parquet")
print("\nParquet wins because:")
print("  1. Column pruning: only 'id_paciente' and 'provincia' are read from disk")
print("  2. Predicate pushdown: row groups where provincia != 'Valencia' are skipped entirely")
print("  3. Better compression: columnar data compresses 5-10x better than CSV")


✅ Data written as CSV and Parquet. Let's compare reads...

CSV (row-oriented):     19,855 rows in 0.828s
Parquet (columnar):     19,855 rows in 0.457s
Speedup:                1.8x faster with Parquet

Parquet wins because:
  1. Column pruning: only 'id_paciente' and 'provincia' are read from disk
  2. Predicate pushdown: row groups where provincia != 'Valencia' are skipped entirely
  3. Better compression: columnar data compresses 5-10x better than CSV


---

## 6. UDFs: The Python Trap & How to Escape It

### 6.1 The Problem with Python UDFs

Spark runs on the **JVM** (Java Virtual Machine). When you use the DataFrame API:

```python
df.filter(F.col("coste") > 100)  # This runs in the JVM → FAST
```

But when you define a **Python UDF**, Spark must:

1. **Serialize** each row from JVM to Python (via `pickle`)
2. Send it to a **Python worker process**
3. Execute your Python function **row by row**
4. **Serialize** the result back to the JVM

This JVM ↔ Python serialization happens **for every single row**. For 1M rows, that's 1M round trips.

### The Mathematical View: Domain → Codomain

A UDF is a function $f: D \to C$ where:
- $D$ is the **domain** (input types: the row values)
- $C$ is the **codomain** (output type: what you must declare explicitly)

Spark requires you to declare the codomain because it needs to know the output schema **at planning time** (before execution). This is the type system being strict — and it's a good thing.

### 6.2 The Solution: Pandas UDFs (Vectorized)

Pandas UDFs use **Apache Arrow** to transfer data in **columnar batches** instead of row by row:

| | Standard UDF | Pandas UDF |
|---|---|---|
| Transfer unit | 1 row at a time | ~10K rows at a time (Arrow batch) |
| Serialization | Python pickle (slow) | Apache Arrow (zero-copy) |
| Python execution | Row by row | Vectorized (NumPy/Pandas) |
| Performance | Slow (🐢) | 3-100x faster (🚀) |


In [13]:
# ============================================================
# Standard Python UDF (slow 🐢)
# ============================================================
from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType

# A "complex" discount calculation
# Business rule: tiered discount based on cost
def calcular_descuento(coste):
    """Calculate a tiered discount:
    - coste <= 100: 5% discount
    - 100 < coste <= 300: 10% discount + flat 5 EUR
    - coste > 300: 15% discount + flat 10 EUR
    """
    if coste is None:
        return 0.0
    if coste <= 100:
        return round(coste * 0.05, 2)
    elif coste <= 300:
        return round(coste * 0.10 + 5.0, 2)
    else:
        return round(coste * 0.15 + 10.0, 2)

# Register the UDF — NOTE: you MUST declare the return type (codomain)
descuento_udf = udf(calcular_descuento, DoubleType())

# Apply the standard UDF
start = time.time()
df_standard_udf = df.withColumn("descuento", descuento_udf(F.col("coste_servicio")))
df_standard_udf.select("coste_servicio", "descuento").show(5)  # Action triggers execution
count_std = df_standard_udf.count()
time_standard = time.time() - start

print(f"\n⏱ Standard UDF: {count_std:,} rows in {time_standard:.2f}s")
print("⚠️  Each row was serialized JVM→Python, processed, then serialized back.")


+--------------+---------+
|coste_servicio|descuento|
+--------------+---------+
|         88.91|     4.45|
|         89.65|     4.48|
|        327.55|    59.13|
|         42.18|     2.11|
|        142.23|    19.22|
+--------------+---------+
only showing top 5 rows


⏱ Standard UDF: 1,000,000 rows in 1.09s
⚠️  Each row was serialized JVM→Python, processed, then serialized back.


In [14]:
# ============================================================
# Pandas UDF / Vectorized UDF (fast 🚀)
# ============================================================
import pandas as pd
from pyspark.sql.functions import pandas_udf

@pandas_udf("double")  # Declare the codomain: output is a double
def calcular_descuento_vectorized(coste: pd.Series) -> pd.Series:
    """Vectorized version: operates on entire Pandas Series at once.
    
    Instead of processing 1 row at a time, this receives ~10K rows
    as a Pandas Series and returns a Pandas Series.
    NumPy vectorized operations are used internally.
    """
    import numpy as np
    
    result = np.where(
        coste <= 100,
        np.round(coste * 0.05, 2),
        np.where(
            coste <= 300,
            np.round(coste * 0.10 + 5.0, 2),
            np.round(coste * 0.15 + 10.0, 2)
        )
    )
    return pd.Series(result)

# Apply the Pandas UDF
start = time.time()
df_pandas_udf = df.withColumn("descuento", calcular_descuento_vectorized(F.col("coste_servicio")))
df_pandas_udf.select("coste_servicio", "descuento").show(5)  # Action triggers execution
count_vec = df_pandas_udf.count()
time_vectorized = time.time() - start

print(f"\n⏱ Pandas UDF: {count_vec:,} rows in {time_vectorized:.2f}s")
print(f"⚡ Speedup: {time_standard / max(time_vectorized, 0.001):.1f}x faster!")
print("\n✅ Data was transferred in Arrow batches and processed with NumPy vectorization.")


+--------------+---------+
|coste_servicio|descuento|
+--------------+---------+
|         88.91|     4.45|
|         89.65|     4.48|
|        327.55|    59.13|
|         42.18|     2.11|
|        142.23|    19.22|
+--------------+---------+
only showing top 5 rows


⏱ Pandas UDF: 1,000,000 rows in 1.68s
⚡ Speedup: 0.6x faster!

✅ Data was transferred in Arrow batches and processed with NumPy vectorization.


In [15]:
# ============================================================
# BONUS: The BEST approach — no UDF at all!
# ============================================================

# If you can express the logic using Spark built-in functions,
# it runs ENTIRELY in the JVM with zero Python overhead.

start = time.time()
df_native = df.withColumn(
    "descuento",
    F.when(F.col("coste_servicio") <= 100,
           F.round(F.col("coste_servicio") * 0.05, 2))
     .when(F.col("coste_servicio") <= 300,
           F.round(F.col("coste_servicio") * 0.10 + 5.0, 2))
     .otherwise(
           F.round(F.col("coste_servicio") * 0.15 + 10.0, 2))
)
df_native.select("coste_servicio", "descuento").show(5)
count_native = df_native.count()
time_native = time.time() - start

print(f"\n⏱ Native Spark (no UDF): {count_native:,} rows in {time_native:.2f}s")
print(f"⚡ vs Standard UDF: {time_standard / max(time_native, 0.001):.1f}x faster")
print(f"⚡ vs Pandas UDF:   {time_vectorized / max(time_native, 0.001):.1f}x faster")

print("\n" + "=" * 60)
print("PERFORMANCE HIERARCHY (fastest to slowest):")
print("=" * 60)
print("1. ✅ Native Spark functions (JVM, no Python)")
print("2. ✅ Pandas UDF (Arrow batches + NumPy)")
print("3. ❌ Standard Python UDF (row-by-row serialization)")
print("\nRule of thumb: ALWAYS try native functions first.")
print("Only use Pandas UDFs when the logic is too complex for built-ins.")
print("NEVER use standard UDFs in production.")


+--------------+---------+
|coste_servicio|descuento|
+--------------+---------+
|         88.91|     4.45|
|         89.65|     4.48|
|        327.55|    59.13|
|         42.18|     2.11|
|        142.23|    19.22|
+--------------+---------+
only showing top 5 rows


⏱ Native Spark (no UDF): 1,000,000 rows in 0.20s
⚡ vs Standard UDF: 5.3x faster
⚡ vs Pandas UDF:   8.2x faster

PERFORMANCE HIERARCHY (fastest to slowest):
1. ✅ Native Spark functions (JVM, no Python)
2. ✅ Pandas UDF (Arrow batches + NumPy)
3. ❌ Standard Python UDF (row-by-row serialization)

Rule of thumb: ALWAYS try native functions first.
Only use Pandas UDFs when the logic is too complex for built-ins.
NEVER use standard UDFs in production.


> **💡 Pro Tip (Interview):** "When asked about UDFs, say: *'I always try to use native Spark functions first. If the logic is complex, I use Pandas UDFs with Apache Arrow for vectorized transfer. I never use standard Python UDFs in production because the row-by-row serialization overhead is prohibitive at scale.'* This shows you understand the JVM/Python boundary problem."


---

## 7. Streaming: Unbounded Datasets & Window Aggregations

### 7.1 What is Structured Streaming?

In batch processing, your dataset is **bounded**: it has a known beginning and end.

$$D_{\text{batch}} = \{x_1, x_2, ..., x_n\} \quad \text{where } n \text{ is finite and known}$$

In streaming, your dataset is **unbounded**: new data arrives continuously, forever.

$$D_{\text{stream}} = \{x_1, x_2, x_3, ...\} \quad \text{where } n \to \infty$$

Spark's Structured Streaming treats a stream as an **infinite, append-only table**. Each micro-batch is like running a batch query on the new rows that arrived since the last batch.

### The Key Concepts:

| Concept | Description | BigQuery Analogy |
|---|---|---|
| **Source** | Where data comes from (Kafka, files, socket) | Streaming Buffer |
| **Sink** | Where results go (console, files, database) | Destination table |
| **Trigger** | How often to process new data | Streaming insert frequency |
| **Watermark** | How late data can arrive before being dropped | N/A (BigQuery handles this internally) |
| **Output Mode** | Append, Complete, or Update | INSERT vs MERGE |

### 7.2 Simulating a Stream


In [16]:
# ============================================================
# DEMO: Structured Streaming with the 'rate' source
# ============================================================

# The 'rate' source generates rows at a fixed rate.
# Each row has: timestamp (Timestamp), value (Long)
# This is perfect for testing streaming logic without external dependencies.

df_stream = (
    spark.readStream
    .format("rate")                    # Built-in source that generates data
    .option("rowsPerSecond", 1000)     # Generate 1000 rows/second
    .option("numPartitions", 4)        # Distribute across 4 partitions
    .load()
)

print("Stream schema:")
df_stream.printSchema()
print(f"Is streaming: {df_stream.isStreaming}")
print("\nNote: No data has been generated yet. This is lazy, just like batch!")


Stream schema:
root
 |-- timestamp: timestamp (nullable = true)
 |-- value: long (nullable = true)

Is streaming: True

Note: No data has been generated yet. This is lazy, just like batch!


In [17]:
# ============================================================
# DEMO: Streaming with simulated health service data
# ============================================================

# Add health-related columns to the rate stream
provincias_array = F.array(*[F.lit(p) for p in PROVINCIAS])

df_health_stream = (
    df_stream
    .withColumn(
        "provincia",
        provincias_array.getItem((F.col("value") % len(PROVINCIAS)).cast(IntegerType()))
    )
    .withColumn(
        "coste_servicio",
        F.round((F.col("value") % 490 + 10).cast(DoubleType()), 2)
    )
    .withColumnRenamed("timestamp", "event_time")
    .drop("value")
)

print("Enhanced stream schema:")
df_health_stream.printSchema()


Enhanced stream schema:
root
 |-- event_time: timestamp (nullable = true)
 |-- provincia: string (nullable = true)
 |-- coste_servicio: double (nullable = true)



### 7.3 Window Aggregation on Streams

In streaming, you can't do a global `groupBy` (the data is infinite!). Instead, you aggregate over **time windows**.

Spark supports three types of windows:

1. **Tumbling Window**: Fixed-size, non-overlapping. Like chopping time into equal blocks.
   - `window(col, "10 minutes")` → [00:00-00:10), [00:10-00:20), ...

2. **Sliding Window**: Fixed-size, overlapping. Like a moving average.
   - `window(col, "10 minutes", "5 minutes")` → windows slide every 5 min

3. **Session Window**: Variable-size, based on activity gaps.
   - Groups events that are close together in time


In [18]:
# ============================================================
# DEMO: Window Aggregation — Average cost per 10-second window
# ============================================================

# Watermark: tells Spark how late data can arrive.
# Data arriving more than 30 seconds late will be dropped.
# This is crucial for managing state in streaming.

df_windowed = (
    df_health_stream
    .withWatermark("event_time", "30 seconds")   # Allow 30s late data
    .groupBy(
        F.window("event_time", "10 seconds"),     # 10-second tumbling window
        "provincia"
    )
    .agg(
        F.count("*").alias("num_events"),
        F.round(F.avg("coste_servicio"), 2).alias("avg_coste"),
        F.round(F.sum("coste_servicio"), 2).alias("total_coste")
    )
)

# Write to the memory sink (for demonstration)
query = (
    df_windowed.writeStream
    .outputMode("update")              # Output only updated rows
    .format("memory")                  # Write to in-memory table
    .queryName("health_windows")       # Table name for SQL queries
    .trigger(processingTime="5 seconds")  # Process every 5 seconds
    .start()
)

print("✅ Streaming query started!")
print(f"Query name: {query.name}")
print(f"Query ID: {query.id}")
print(f"Is active: {query.isActive}")
print("\nWaiting 15 seconds for data to accumulate...")


✅ Streaming query started!
Query name: health_windows
Query ID: 9602e874-ea3a-4a63-afed-4eba281c8593
Is active: True

Waiting 15 seconds for data to accumulate...


In [19]:
# ============================================================
# Query the streaming results
# ============================================================

# Wait for some data to accumulate
time.sleep(15)

# Query the in-memory table
print("=== Streaming Window Aggregation Results ===")
print("Each row represents the average cost in a 10-second window for a province:\n")

spark.sql("""
    SELECT 
        window.start AS window_start,
        window.end AS window_end,
        provincia,
        num_events,
        avg_coste,
        total_coste
    FROM health_windows
    ORDER BY window_start DESC, total_coste DESC
    LIMIT 20
""").show(truncate=False)

# Check streaming query status
print(f"\nStreaming status: {query.status}")
print(f"Recent progress:")
if query.recentProgress:
    latest = query.recentProgress[-1]
    print(f"  Input rows/sec: {latest.get('inputRowsPerSecond', 'N/A')}")
    print(f"  Processed rows/sec: {latest.get('processedRowsPerSecond', 'N/A')}")


=== Streaming Window Aggregation Results ===
Each row represents the average cost in a 10-second window for a province:

+-------------------+-------------------+----------+----------+---------+-----------+
|window_start       |window_end         |provincia |num_events|avg_coste|total_coste|
+-------------------+-------------------+----------+----------+---------+-----------+
|2026-02-14 13:07:50|2026-02-14 13:08:00|Jaen      |87        |261.18   |22723.0    |
|2026-02-14 13:07:50|2026-02-14 13:08:00|Lleida    |87        |260.18   |22636.0    |
|2026-02-14 13:07:50|2026-02-14 13:08:00|Avila     |87        |259.92   |22613.0    |
|2026-02-14 13:07:50|2026-02-14 13:08:00|Tarragona |87        |259.18   |22549.0    |
|2026-02-14 13:07:50|2026-02-14 13:08:00|Segovia   |87        |258.92   |22526.0    |
|2026-02-14 13:07:50|2026-02-14 13:08:00|Logrono   |87        |258.66   |22503.0    |
|2026-02-14 13:07:50|2026-02-14 13:08:00|Leon      |87        |258.18   |22462.0    |
|2026-02-14 13:07:5

In [20]:
# Stop the streaming query (important: don't leave it running!)
query.stop()
print("✅ Streaming query stopped.")


✅ Streaming query stopped.


### 7.4 Approximate Counting: HyperLogLog

When processing huge (or infinite) datasets, exact counting of distinct values is expensive because you need to keep **every unique value in memory**.

For a set of $n$ elements with $d$ distinct values:
- **Exact count:** $O(d)$ memory (store every unique value in a hash set)
- **Approximate count (HyperLogLog):** $O(\log \log d)$ memory (!)

### How HyperLogLog Works (Intuition)

The algorithm is based on a beautiful probabilistic insight:

1. **Hash** each element to get a uniformly distributed binary string
2. **Count leading zeros** in each hash: if you see $k$ leading zeros, the probability is $\frac{1}{2^k}$
3. **Maximum leading zeros** observed tells you about the cardinality: if max leading zeros = $k$, then $d \approx 2^k$

Think of it like coin flips: if someone tells you they flipped 20 heads in a row, you'd estimate they flipped the coin roughly $2^{20} \approx 1M$ times.

The "Hyper" in HyperLogLog comes from using **harmonic mean** across multiple buckets to reduce variance:

$$\hat{n} = \frac{\alpha_m \cdot m^2}{\sum_{j=1}^{m} 2^{-M_j}}$$

where $m$ is the number of buckets, $M_j$ is the max leading zeros in bucket $j$, and $\alpha_m$ is a bias correction constant.

Spark's `approx_count_distinct()` uses HyperLogLog internally with a configurable relative standard deviation (default 5%).


In [21]:
# ============================================================
# DEMO: Exact vs Approximate Distinct Count
# ============================================================

# --- Exact count (requires full shuffle + hash set in memory) ---
start = time.time()
exact_count = df.select(F.countDistinct("id_paciente")).collect()[0][0]
time_exact = time.time() - start

# --- Approximate count (HyperLogLog, much cheaper) ---
start = time.time()
approx_count = df.select(F.approx_count_distinct("id_paciente", rsd=0.05)).collect()[0][0]
time_approx = time.time() - start

error_pct = abs(exact_count - approx_count) / exact_count * 100

print("=== Exact vs Approximate Distinct Count ===")
print(f"Exact count:       {exact_count:>12,}   ({time_exact:.3f}s)")
print(f"Approx count:      {approx_count:>12,}   ({time_approx:.3f}s)")
print(f"Error:             {error_pct:>11.2f}%")
print(f"Speedup:           {time_exact / max(time_approx, 0.001):>11.1f}x")
print(f"\nHyperLogLog uses O(log log n) memory vs O(n) for exact counting.")
print(f"At scale (billions of rows), this difference is the key to real-time analytics.")


=== Exact vs Approximate Distinct Count ===
Exact count:          1,000,000   (0.660s)
Approx count:           943,039   (0.354s)
Error:                    5.70%
Speedup:                   1.9x

HyperLogLog uses O(log log n) memory vs O(n) for exact counting.
At scale (billions of rows), this difference is the key to real-time analytics.


In [22]:
# ============================================================
# DEMO: Approximate count per province
# ============================================================

# Compare exact vs approximate grouped counts
comparison = (
    df.groupBy("provincia")
    .agg(
        F.countDistinct("id_paciente").alias("exact_distinct"),
        F.approx_count_distinct("id_paciente", rsd=0.05).alias("approx_distinct")
    )
    .withColumn(
        "error_pct",
        F.round(
            F.abs(F.col("exact_distinct") - F.col("approx_distinct")) 
            / F.col("exact_distinct") * 100, 2
        )
    )
    .orderBy(F.desc("exact_distinct"))
)

print("Exact vs Approximate distinct patients per province:")
comparison.show(15, truncate=False)

# Summary stats on error
print("Error statistics across all provinces:")
comparison.select(
    F.round(F.avg("error_pct"), 2).alias("mean_error_%"),
    F.round(F.max("error_pct"), 2).alias("max_error_%"),
    F.round(F.min("error_pct"), 2).alias("min_error_%")
).show()


Exact vs Approximate distinct patients per province:
+---------+--------------+---------------+---------+
|provincia|exact_distinct|approx_distinct|error_pct|
+---------+--------------+---------------+---------+
|Huelva   |20255         |19325          |4.59     |
|Castellon|20212         |20267          |0.27     |
|Caceres  |20212         |20882          |3.31     |
|Jaen     |20197         |18538          |8.21     |
|Madrid   |20185         |21234          |5.2      |
|Avila    |20179         |19795          |1.9      |
|Lugo     |20160         |20753          |2.94     |
|Barcelona|20147         |19582          |2.8      |
|Bilbao   |20131         |18522          |7.99     |
|Albacete |20102         |18162          |9.65     |
|Logrono  |20086         |19814          |1.35     |
|Lleida   |20086         |20791          |3.51     |
|Girona   |20073         |20712          |3.18     |
|Zamora   |20072         |20207          |0.67     |
|Tarragona|20070         |20155          |0.42

> **💡 Pro Tip (Interview):** "When asked about counting distinct values at scale, mention HyperLogLog. It uses $O(\log \log n)$ memory with ~2-5% error. In Spark, it's `approx_count_distinct()`. In BigQuery, `APPROX_COUNT_DISTINCT()` uses the same algorithm. For real-time dashboards, this trade-off (speed vs. precision) is almost always worth it."


---

## 8. Summary & Cheat Sheet

### Concept Map

```
┌───────────────────────────────────────────────────────────────────────┐
│                        SPARK ARCHITECTURE                            │
├───────────────────────────────────────────────────────────────────────┤
│  DataFrame API / SQL                                                 │
│       │                                                              │
│       ▼                                                              │
│  Catalyst Optimizer  ──────>  Optimized Logical Plan                 │
│       │                                                              │
│       ▼                                                              │
│  Tungsten Engine  ─────────>  Physical Plan + Code Gen               │
│       │                                                              │
│       ▼                                                              │
│  DAG Scheduler  ───────────>  Jobs → Stages → Tasks                  │
│       │                                                              │
│       ▼                                                              │
│  Task Scheduler  ──────────>  Assigns tasks to executors             │
│       │                                                              │
│       ▼                                                              │
│  Executors (JVM)  ─────────>  Process partitions in parallel         │
└───────────────────────────────────────────────────────────────────────┘
```

### Quick Reference

| What | Command | Notes |
|---|---|---|
| Start session | `SparkSession.builder.getOrCreate()` | Entry point to everything |
| Check partitions | `df.rdd.getNumPartitions()` | 1 partition = 1 task |
| View plan | `df.explain(mode="formatted")` | Look for Exchange = shuffle |
| Reduce partitions | `df.coalesce(n)` | No full shuffle (narrow) |
| Increase partitions | `df.repartition(n)` | Full shuffle (wide) |
| Cache data | `df.cache()` or `df.persist()` | Keep in memory for reuse |
| Approx distinct | `F.approx_count_distinct(col, rsd)` | HyperLogLog, O(log log n) |
| Vectorized UDF | `@pandas_udf("type")` | Use Arrow, not pickle |
| Stream window | `F.window(col, "10 minutes")` | Tumbling window |
| Watermark | `.withWatermark(col, delay)` | Drop late data |

### The Golden Rules of Spark Optimization

1. **Minimize shuffles** — Every `groupBy`, `join`, `distinct` triggers a shuffle
2. **Use columnar formats** — Parquet > CSV, always
3. **Avoid Python UDFs** — Native functions > Pandas UDFs > standard UDFs
4. **Right-size partitions** — Aim for 128MB-256MB per partition
5. **Cache strategically** — Cache DataFrames you reuse, unpersist when done
6. **Let Catalyst work** — Use the DataFrame API (not RDDs) so the optimizer can help
7. **Filter early** — Push filters before joins and aggregations

---

*Thank you for reading! If this tutorial helped you, please upvote and leave a comment. Happy Sparking!* 🚀


In [23]:
# ============================================================
# Cleanup: Stop the SparkSession
# ============================================================

# Clean up temp files
import shutil
try:
    shutil.rmtree(TEMP_DIR)
except:
    pass

spark.stop()
print("✅ SparkSession stopped. Resources released.")
print("\nThanks for following along! Key takeaways:")
print("  1. Spark is lazy: transformations build a DAG, actions trigger execution")
print("  2. Shuffles are expensive: minimize them")
print("  3. Columnar storage (Parquet) = matrix transpose for optimal reads")
print("  4. Native functions > Pandas UDFs > Python UDFs")
print("  5. HyperLogLog for approximate counting at scale")
print("  6. Streaming = infinite table processed in micro-batches")


✅ SparkSession stopped. Resources released.

Thanks for following along! Key takeaways:
  1. Spark is lazy: transformations build a DAG, actions trigger execution
  2. Shuffles are expensive: minimize them
  3. Columnar storage (Parquet) = matrix transpose for optimal reads
  4. Native functions > Pandas UDFs > Python UDFs
  5. HyperLogLog for approximate counting at scale
  6. Streaming = infinite table processed in micro-batches
